# TP N°2 – Predicción de la demanda de EcoBici por estación y franja horaria con RNA
**Inteligencia Artificial – UTN FRBA – 2C 2026 – Grupo N° 1**

Este notebook implementa de punta a punta lo planteado en la entrega parcial:

1. Descarga de datos públicos (recorridos y estaciones de BA Data, clima horario de Open-Meteo, feriados de ArgentinaDatos).
2. Limpieza, agregación por estación y hora, y construcción de atributos.
3. Distribución temporal: entrenamiento (ene-2024 a may-2025), validación (jun–jul 2025) y prueba (ago–dic 2025).
4. Entrenamientos:
   - **Modelo A** – MLP BP de regresión (retiros y devoluciones de la hora siguiente).
   - **Modelo A2** – variante del MLP de regresión con otra topología y regularización.
   - **Modelo B** – MLP BP de clasificación (nivel de retiros: baja / media / alta).
   - **Modelo C** – ConvNet 1D sobre las últimas 24 horas de cada estación.
5. Métricas sobre entrenamiento y prueba, gráficos de loss, comparación contra métodos simples (línea base).

**Cómo correrlo:**
1. Descargar a mano (desde el navegador) los recorridos 2024 y 2025 y el CSV de estaciones de BA Data.
   El portal tiene protección anti-bots y no permite descargarlos desde Colab.
2. Subir esos archivos a una carpeta de Google Drive llamada `TP2_EcoBici` (en *Mi unidad*). No hace falta renombrarlos.
3. *Entorno de ejecución → Ejecutar todo* y aceptar el permiso de acceso a Drive.

Funciona en CPU (la GPU es opcional). Al final se descargan `resultados_tp2.zip` (métricas, gráficos y resumen para el informe)
y `datos_tp2.zip` (CSV para el repositorio).

## 0. Configuración

In [ ]:
import os, io, re, json, time, zipfile, unicodedata, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
SEED = 42
keras.utils.set_random_seed(SEED)

CONFIG = dict(
    anios=[2024, 2025],
    fecha_inicio='2024-01-01',
    fecha_fin='2025-12-31',
    fin_entrenamiento='2025-05-31',   # entrenamiento: ene-2024 a may-2025
    fin_validacion='2025-07-31',      # validación: jun–jul 2025 ; prueba: ago–dic 2025
    n_estaciones=30,
    lat=-34.6037, lon=-58.3816,       # centro de CABA (para el clima)
    epocas=60, batch=512, paciencia=6,
)

DIR_RAW, DIR_PROC = 'data/raw', 'data/processed'
DIR_RES, DIR_FIG = 'resultados', 'resultados/figuras'
for d in [DIR_RAW, DIR_PROC, DIR_RES, DIR_FIG]:
    os.makedirs(d, exist_ok=True)

def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return s.strip().strip('"').strip("'").lower()

RESUMEN = {'config': CONFIG}   # se completa a lo largo del notebook; sirve para escribir el informe
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Datos
### 1.0 Copia de los archivos desde Google Drive

In [ ]:
CARPETA_DRIVE = '/content/drive/MyDrive/TP2_EcoBici'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('No se está en Colab o no se pudo montar Drive:', e)

import shutil, glob
if os.path.isdir(CARPETA_DRIVE):
    for f in sorted(glob.glob(os.path.join(CARPETA_DRIVE, '*'))):
        base = os.path.basename(f)
        nl, ext = norm(base), os.path.splitext(base)[1].lower()
        anio = next((a for a in CONFIG['anios'] if str(a) in base), None)
        if 'estacion' in nl and ext == '.csv':
            destino = 'estaciones.csv'
        elif anio and ext in ('.zip', '.csv'):
            destino = f'recorridos_{anio}{ext}'
        else:
            print('  se ignora:', base)
            continue
        shutil.copy(f, os.path.join(DIR_RAW, destino))
        print(f'  copiado: {base}  →  data/raw/{destino}')
else:
    print(f'>> No existe la carpeta {CARPETA_DRIVE}. Creala en Drive y subí ahí los archivos.')
print('Contenido de data/raw:', sorted(os.listdir(DIR_RAW)))

### 1.1 Recorridos (si no se subieron a Drive, se intenta la descarga automática)

In [ ]:
CKAN = 'https://data.buenosaires.gob.ar/api/3/action/package_show?id={}'
PAGINA = 'https://data.buenosaires.gob.ar/dataset/{}'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/124.0 Safari/537.36'}

def recursos_ckan(dataset_id):
    r = None
    try:
        r = requests.get(CKAN.format(dataset_id), headers=HEADERS, timeout=60)
        return r.json()['result']['resources']
    except Exception as e:
        print(f'  API no disponible ({e}); status={getattr(r, "status_code", "?")}')
    r = requests.get(PAGINA.format(dataset_id), headers=HEADERS, timeout=60)
    links = sorted(set(re.findall(r'href="([^"]+\.(?:zip|csv))"', r.text, flags=re.I)))
    if not links:
        raise RuntimeError('el portal bloquea la descarga automática (protección anti-bots)')
    return [{'name': os.path.basename(u), 'url': u, 'format': u.rsplit('.', 1)[-1]} for u in links]

def descargar(url, destino):
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        print('  ya existe:', destino)
        return destino
    with requests.get(url, headers=HEADERS, stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(destino, 'wb') as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    print(f'  descargado: {destino} ({os.path.getsize(destino)/1e6:.1f} MB)')
    return destino

def recorridos_locales():
    return sorted(os.path.join(DIR_RAW, f) for f in os.listdir(DIR_RAW) if f.startswith('recorridos_'))

archivos_recorridos = recorridos_locales()
if len(archivos_recorridos) < len(CONFIG['anios']):
    print('Faltan archivos de recorridos en data/raw: se intenta la descarga automática...')
    try:
        recursos = recursos_ckan('bicicletas-publicas')
        for anio in CONFIG['anios']:
            cands = [r for r in recursos
                     if str(anio) in (r.get('name', '') + r.get('url', ''))
                     and 'recorrido' in (r.get('name', '') + r.get('url', '')).lower()]
            if cands:
                url = cands[0]['url']
                ext = os.path.splitext(url.split('?')[0])[1].lower() or '.zip'
                descargar(url, os.path.join(DIR_RAW, f'recorridos_{anio}{ext}'))
    except Exception as e:
        print('>> Falló la descarga automática:', e)
    archivos_recorridos = recorridos_locales()

print('Archivos de recorridos a usar:', archivos_recorridos)
assert archivos_recorridos, 'No hay archivos de recorridos: subilos a la carpeta TP2_EcoBici de Drive (ver instrucciones arriba).'

### 1.2 Lectura y limpieza de recorridos

In [ ]:
def _csvs(ruta):
    """Lista de (nombre, función que abre el archivo) para un CSV suelto o los CSV dentro de un ZIP."""
    if ruta.lower().endswith('.zip'):
        z = zipfile.ZipFile(ruta)
        return [(n, (lambda n=n: z.open(n))) for n in z.namelist() if n.lower().endswith('.csv')]
    return [(ruta, (lambda: open(ruta, 'rb')))]

def _buscar(cols, *claves, excluir=()):
    for c in cols:
        if all(k in c for k in claves) and not any(x in c for x in excluir):
            return c
    return None

def leer_recorridos(ruta):
    partes = []
    for nombre, abrir in _csvs(ruta):
        with abrir() as f:
            cab = f.readline().decode('utf-8', errors='ignore')
        sep = ';' if cab.count(';') > cab.count(',') else ','
        cols = [norm(c) for c in cab.strip().split(sep)]
        m = {
            'fecha_origen':  _buscar(cols, 'fecha', 'origen'),
            'fecha_destino': _buscar(cols, 'fecha', 'destino'),
            'est_origen':    _buscar(cols, 'id', 'estacion', 'origen', excluir=('nombre', 'direccion')),
            'est_destino':   _buscar(cols, 'id', 'estacion', 'destino', excluir=('nombre', 'direccion')),
            'nombre_origen': _buscar(cols, 'nombre', 'origen'),
            'lat_origen':    _buscar(cols, 'lat', 'origen'),
            'lon_origen':    _buscar(cols, 'long', 'origen'),
        }
        faltan = [k for k in ['fecha_origen', 'fecha_destino', 'est_origen', 'est_destino'] if m[k] is None]
        if faltan:
            raise ValueError(f'{nombre}: no se encontraron las columnas {faltan}. Columnas: {cols}')
        inv = {v: k for k, v in m.items() if v is not None}
        for enc in ['utf-8', 'latin-1']:
            try:
                with abrir() as f:
                    df = pd.read_csv(f, sep=sep, encoding=enc, usecols=lambda c: norm(c) in inv,
                                     dtype=str, low_memory=False)
                break
            except UnicodeDecodeError:
                continue
        df.columns = [inv[norm(c)] for c in df.columns]
        print(f'  {nombre}: {len(df):,} filas')
        partes.append(df)
    return pd.concat(partes, ignore_index=True)

def a_id(s):
    # los identificadores pueden venir como "5", "005" o "5BAEcobici": se toma la parte numérica
    return pd.to_numeric(s.astype(str).str.extract(r'(\d+)')[0], errors='coerce')

crudos = pd.concat([leer_recorridos(r) for r in archivos_recorridos], ignore_index=True)
n_crudos = len(crudos)
for c in ['fecha_origen', 'fecha_destino']:
    crudos[c] = pd.to_datetime(crudos[c], errors='coerce', format='mixed')
for c in ['est_origen', 'est_destino']:
    crudos[c] = a_id(crudos[c])
for c in ['lat_origen', 'lon_origen']:
    if c in crudos:
        crudos[c] = pd.to_numeric(crudos[c].str.replace(',', '.'), errors='coerce')

dur = (crudos['fecha_destino'] - crudos['fecha_origen']).dt.total_seconds()
filtros = {
    'sin_fecha': crudos['fecha_origen'].isna() | crudos['fecha_destino'].isna(),
    'sin_estacion': crudos['est_origen'].isna() | crudos['est_destino'].isna(),
    'duracion_nula_o_negativa': dur <= 0,
    'duracion_mayor_24h': dur > 86400,
}
descarte = np.zeros(len(crudos), dtype=bool)
RESUMEN['limpieza'] = {'recorridos_crudos': int(n_crudos)}
for k, msk in filtros.items():
    nuevos = msk.fillna(False).values & ~descarte
    RESUMEN['limpieza'][k] = int(nuevos.sum())
    descarte |= msk.fillna(False).values
viajes = crudos[~descarte].copy()
del crudos
RESUMEN['limpieza']['recorridos_validos'] = int(len(viajes))
print(json.dumps(RESUMEN['limpieza'], indent=2))
print('Rango de fechas:', viajes['fecha_origen'].min(), '→', viajes['fecha_origen'].max())

### 1.3 Estaciones (ubicación y capacidad)

In [ ]:
info_est = None
try:
    ruta = os.path.join(DIR_RAW, 'estaciones.csv')
    if not os.path.exists(ruta):
        rec = recursos_ckan('estaciones-bicicletas-publicas')
        csvs = [r for r in rec if r.get('format', '').lower() == 'csv' or r.get('url', '').lower().endswith('.csv')]
        ruta = descargar(csvs[0]['url'], ruta)
    try:
        est = pd.read_csv(ruta, sep=None, engine='python', encoding='utf-8')
    except UnicodeDecodeError:
        est = pd.read_csv(ruta, sep=None, engine='python', encoding='latin-1')
    est.columns = [norm(c) for c in est.columns]
    print('Columnas del archivo de estaciones:', list(est.columns))
    col_id = next((c for c in ['id', 'id_estacion', 'nro_est', 'numero', 'numero_estacion', 'station_id'] if c in est.columns),
                  next((c for c in est.columns if 'id' in c), None))
    col_cap = next((c for c in est.columns if 'capac' in c or 'anclaje' in c), None)
    col_nom = next((c for c in est.columns if 'nombre' in c or c == 'name'), None)
    info_est = pd.DataFrame({'estacion': a_id(est[col_id])})
    info_est['capacidad'] = pd.to_numeric(est[col_cap], errors='coerce') if col_cap else np.nan
    info_est['nombre'] = est[col_nom] if col_nom else None
    info_est = info_est.dropna(subset=['estacion']).drop_duplicates('estacion').set_index('estacion')
except Exception as e:
    print('>> No se pudo obtener el archivo de estaciones:', e)

USAR_CAPACIDAD = info_est is not None and info_est['capacidad'].notna().any()
RESUMEN['capacidad_disponible'] = bool(USAR_CAPACIDAD)
print('¿Se usa la capacidad como atributo?', USAR_CAPACIDAD)

### 1.4 Clima horario (Open-Meteo) y feriados (ArgentinaDatos)

In [ ]:
url = ('https://archive-api.open-meteo.com/v1/archive'
       f"?latitude={CONFIG['lat']}&longitude={CONFIG['lon']}"
       f"&start_date={CONFIG['fecha_inicio']}&end_date={CONFIG['fecha_fin']}"
       '&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m'
       '&timezone=America%2FArgentina%2FBuenos_Aires')
h = requests.get(url, timeout=120).json()['hourly']
clima = pd.DataFrame({
    'fecha_hora': pd.to_datetime(h['time']),
    'temperatura': h['temperature_2m'],
    'humedad': h['relative_humidity_2m'],
    'precipitacion': h['precipitation'],
    'viento': h['wind_speed_10m'],
}).set_index('fecha_hora').ffill().bfill()
clima.to_csv(os.path.join(DIR_RAW, 'clima_horario.csv'))
print('Clima:', clima.shape, '| horas con lluvia:', int((clima.precipitacion > 0).sum()))

feriados = []
try:
    for a in CONFIG['anios']:
        feriados += requests.get(f'https://api.argentinadatos.com/v1/feriados/{a}', timeout=60).json()
    feriados = pd.DataFrame(feriados)
    feriados['fecha'] = pd.to_datetime(feriados['fecha'])
except Exception as e:
    print('>> ArgentinaDatos no respondió, se usa la librería holidays:', e)
    os.system('pip -q install holidays')
    import holidays
    fa = holidays.country_holidays('AR', years=CONFIG['anios'])
    feriados = pd.DataFrame({'fecha': pd.to_datetime(list(fa.keys())), 'nombre': list(fa.values())})
feriados.to_csv(os.path.join(DIR_RAW, 'feriados.csv'), index=False)
DIAS_FERIADO = set(feriados['fecha'].dt.normalize())
print('Feriados cargados:', len(DIAS_FERIADO))

## 2. Preparación de los datos
### 2.1 Selección de estaciones y agregación por estación y hora

In [ ]:
ini = pd.Timestamp(CONFIG['fecha_inicio'])
fin = pd.Timestamp(CONFIG['fecha_fin']) + pd.Timedelta(hours=23)
horas = pd.date_range(ini, fin, freq='h')

en_rango_o = viajes['fecha_origen'].between(ini, fin + pd.Timedelta(minutes=59, seconds=59))
en_rango_d = viajes['fecha_destino'].between(ini, fin + pd.Timedelta(minutes=59, seconds=59))

# estaciones activas durante todo el período: con retiros en cada uno de los meses
meses = viajes.loc[en_rango_o, 'fecha_origen'].dt.to_period('M')
meses_activos = meses.groupby(viajes.loc[en_rango_o, 'est_origen']).nunique()
n_meses = len(pd.period_range(ini, fin, freq='M'))
elegibles = meses_activos[meses_activos == n_meses].index
total = (viajes.loc[en_rango_o, 'est_origen'].value_counts()
         .add(viajes.loc[en_rango_d, 'est_destino'].value_counts(), fill_value=0))
top = total.loc[total.index.isin(elegibles)].nlargest(CONFIG['n_estaciones'])
ESTACIONES = sorted(int(e) for e in top.index)
print(f'Estaciones activas los {n_meses} meses: {len(elegibles)} | seleccionadas: {len(ESTACIONES)}')
if len(ESTACIONES) < CONFIG['n_estaciones']:
    print('>> ATENCIÓN: hay menos estaciones activas todo el período de lo previsto. '
          'Revisar si cambiaron los identificadores entre años.')

vo = viajes[en_rango_o & viajes['est_origen'].isin(ESTACIONES)]
vd = viajes[en_rango_d & viajes['est_destino'].isin(ESTACIONES)]
ret = vo.groupby([vo['est_origen'].astype(int), vo['fecha_origen'].dt.floor('h')]).size().rename('retiros')
dev = vd.groupby([vd['est_destino'].astype(int), vd['fecha_destino'].dt.floor('h')]).size().rename('devoluciones')
ret.index.names = dev.index.names = ['estacion', 'fecha_hora']

idx = pd.MultiIndex.from_product([ESTACIONES, horas], names=['estacion', 'fecha_hora'])
panel = pd.DataFrame(index=idx).join(ret).join(dev).fillna(0).astype(int).reset_index()

# atributos de cada estación: nombre y ubicación (desde los recorridos), capacidad (desde el archivo de estaciones)
attr = pd.DataFrame(index=pd.Index(ESTACIONES, name='estacion'))
if 'lat_origen' in vo:
    g = vo.groupby(vo['est_origen'].astype(int))
    attr['latitud'] = g['lat_origen'].median()
    attr['longitud'] = g['lon_origen'].median()
if 'nombre_origen' in vo:
    attr['nombre'] = vo.groupby(vo['est_origen'].astype(int))['nombre_origen'].agg(lambda s: s.mode().iat[0])
if USAR_CAPACIDAD:
    attr['capacidad'] = info_est['capacidad'].reindex(attr.index)
    if attr['capacidad'].isna().any():
        print('>> Estaciones sin capacidad (se completan con la mediana):', list(attr.index[attr['capacidad'].isna()]))
        attr['capacidad'] = attr['capacidad'].fillna(attr['capacidad'].median())
attr['viajes_totales'] = top.reindex(attr.index).astype(int)
display(attr)

RESUMEN['estaciones'] = attr.reset_index().to_dict(orient='records')
RESUMEN['panel'] = dict(filas=int(len(panel)), horas=int(len(horas)), estaciones=len(ESTACIONES),
                        pct_horas_sin_retiros=float((panel.retiros == 0).mean() * 100),
                        retiros_media=float(panel.retiros.mean()), retiros_max=int(panel.retiros.max()),
                        devoluciones_media=float(panel.devoluciones.mean()), devoluciones_max=int(panel.devoluciones.max()))
print(RESUMEN['panel'])

### 2.2 Construcción de atributos, objetivos y distribución temporal

In [ ]:
p = panel.sort_values(['estacion', 'fecha_hora']).reset_index(drop=True)
p['pos'] = p.groupby('estacion').cumcount()          # posición horaria dentro de la estación
gr = p.groupby('estacion')
for col, corto in [('retiros', 'retiros'), ('devoluciones', 'devol')]:
    p[f'{corto}_h-1'] = gr[col].shift(1)     # hora anterior
    p[f'{corto}_d-1'] = gr[col].shift(24)    # misma hora del día anterior
    p[f'{corto}_s-1'] = gr[col].shift(168)   # misma hora de la semana anterior

fh = p['fecha_hora']
p['hora'] = fh.dt.hour
p['dia_semana'] = fh.dt.dayofweek            # 0 = lunes
p['mes'] = fh.dt.month
p['fin_de_semana'] = (p['dia_semana'] >= 5).astype(int)
p['feriado'] = fh.dt.normalize().isin(DIAS_FERIADO).astype(int)
for c, per in [('hora', 24), ('dia_semana', 7), ('mes', 12)]:
    p[f'{c}_sin'] = np.sin(2 * np.pi * p[c] / per)
    p[f'{c}_cos'] = np.cos(2 * np.pi * p[c] / per)
p = p.join(clima, on='fecha_hora')
p = p.join(attr.drop(columns=['nombre', 'viajes_totales'], errors='ignore'), on='estacion')

fe, fv = pd.Timestamp(CONFIG['fin_entrenamiento']) + pd.Timedelta(hours=23), pd.Timestamp(CONFIG['fin_validacion']) + pd.Timedelta(hours=23)
p['conjunto'] = np.where(fh <= fe, 'entrenamiento', np.where(fh <= fv, 'validacion', 'prueba'))

# Objetivo de clasificación (modelo B): umbrales por estación calculados solo con horas con actividad de entrenamiento
act = p[(p.conjunto == 'entrenamiento') & (p.retiros > 0)]
umbrales = act.groupby('estacion')['retiros'].quantile([1/3, 2/3]).unstack()
umbrales.columns = ['p33', 'p66']
p = p.join(umbrales, on='estacion')
p['nivel_demanda'] = np.select([p.retiros <= p.p33, p.retiros <= p.p66], [0, 1], 2)   # 0 baja, 1 media, 2 alta
CLASES = ['baja', 'media', 'alta']

p_full = p.copy()                        # se usa para armar las secuencias de la ConvNet
dfm = p[p['pos'] >= 168].reset_index(drop=True)   # se descarta la primera semana (sin s-1)

RESUMEN['umbrales_clases'] = umbrales.reset_index().to_dict(orient='records')
RESUMEN['conjuntos'] = {}
for c in ['entrenamiento', 'validacion', 'prueba']:
    s = dfm[dfm.conjunto == c]
    RESUMEN['conjuntos'][c] = dict(
        ejemplos=int(len(s)), porcentaje=round(100 * len(s) / len(dfm), 2),
        desde=str(s.fecha_hora.min()), hasta=str(s.fecha_hora.max()),
        distribucion_clases={CLASES[k]: int(v) for k, v in s.nivel_demanda.value_counts().sort_index().items()})
print(json.dumps(RESUMEN['conjuntos'], indent=2, ensure_ascii=False))

### 2.3 Exportación de los datos a CSV (para el repositorio)

In [ ]:
cols_csv = ['estacion', 'fecha_hora', 'hora', 'dia_semana', 'mes', 'fin_de_semana', 'feriado',
            'temperatura', 'humedad', 'precipitacion', 'viento'] + \
           [c for c in ['latitud', 'longitud', 'capacidad'] if c in dfm] + \
           ['retiros_h-1', 'devol_h-1', 'retiros_d-1', 'devol_d-1', 'retiros_s-1', 'devol_s-1',
            'retiros', 'devoluciones', 'nivel_demanda']
for c in ['entrenamiento', 'validacion', 'prueba']:
    ruta = os.path.join(DIR_PROC, f'dataset_{c}.csv')
    dfm.loc[dfm.conjunto == c, cols_csv].to_csv(ruta, index=False, float_format='%.4f')
    print(f'{ruta}: {os.path.getsize(ruta)/1e6:.1f} MB')
print('Nota: GitHub no acepta archivos de más de 100 MB; si alguno lo supera, subirlo comprimido o con Git LFS.')

### 2.4 Normalización y matrices de entrada

In [ ]:
LAGS = ['retiros_h-1', 'devol_h-1', 'retiros_d-1', 'devol_d-1', 'retiros_s-1', 'devol_s-1']
NUM = [c for c in ['latitud', 'longitud', 'capacidad'] if c in dfm] + ['temperatura', 'humedad', 'precipitacion', 'viento']
CIC = ['hora_sin', 'hora_cos', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos']
BIN = ['fin_de_semana', 'feriado']

tr, va, te = (dfm.conjunto == 'entrenamiento').values, (dfm.conjunto == 'validacion').values, (dfm.conjunto == 'prueba').values

def matriz(df, cols_num, cols_lag, ref=None):
    num = df[cols_num].astype(float)
    lag = np.log1p(df[cols_lag].astype(float)) if cols_lag else pd.DataFrame(index=df.index)
    bloque = pd.concat([num, lag], axis=1)
    if ref is None:   # parámetros de normalización: solo del conjunto de entrenamiento
        ref = (bloque[tr].mean(), bloque[tr].std().replace(0, 1))
    bloque = (bloque - ref[0]) / ref[1]
    oh = pd.get_dummies(df['estacion'], prefix='est').astype(float)
    X = pd.concat([bloque, df[CIC + BIN].astype(float), oh], axis=1)
    return X.values.astype('float32'), list(X.columns), ref

X, COLS_X, _ = matriz(dfm, NUM, LAGS)
Y_reg = dfm[['retiros', 'devoluciones']].values.astype('float32')     # en cantidades
Y_cls = dfm['nivel_demanda'].values.astype('int32')
print('Atributos de entrada (MLP):', X.shape[1])
print(COLS_X)
RESUMEN['atributos_entrada_mlp'] = COLS_X

## 3. Funciones auxiliares (entrenamiento, métricas y gráficos)

In [ ]:
OBJ = ['retiros', 'devoluciones']
METRICAS, INFO_ENTRENAMIENTO = {}, {}

def metricas_regresion(y, yp):
    out = {}
    for i, n in enumerate(OBJ):
        r, pr = y[:, i].astype(float), yp[:, i].astype(float)
        e, pos = np.abs(r - pr), r > 0
        out[n] = dict(
            error_absoluto_medio=float(e.mean()),
            raiz_error_cuadratico_medio=float(np.sqrt(((r - pr) ** 2).mean())),
            error_relativo_agregado_pct=float(100 * e.sum() / r.sum()),               # suma|e| / suma(real)
            error_relativo_medio_pct_horas_con_demanda=float(100 * (e[pos] / r[pos]).mean()),
            promedio_real=float(r.mean()), promedio_predicho=float(pr.mean()))
    return out

def mostrar_regresion(nombre, m):
    filas = []
    for conj, d in m.items():
        for obj, v in d.items():
            filas.append({'modelo': nombre, 'conjunto': conj, 'objetivo': obj, **v})
    t = pd.DataFrame(filas).round(3)
    display(t)
    return t

def metricas_clasificacion(y, yp):
    cm = confusion_matrix(y, yp, labels=[0, 1, 2])
    pr, rc, f1, sop = precision_recall_fscore_support(y, yp, labels=[0, 1, 2], zero_division=0)
    return dict(exactitud=float(accuracy_score(y, yp)), matriz_confusion=cm.tolist(),
                por_clase={CLASES[k]: dict(precision_pct=float(100 * pr[k]), recuperacion_pct=float(100 * rc[k]),
                                           f1=float(f1[k]), soporte=int(sop[k])) for k in range(3)})

def graficar_loss(h, nombre, titulo):
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(h.history['loss']) + 1), h.history['loss'], label='Entrenamiento')
    plt.plot(range(1, len(h.history['val_loss']) + 1), h.history['val_loss'], label='Validación')
    best = int(np.argmin(h.history['val_loss'])) + 1
    plt.axvline(best, ls='--', c='gray', lw=1, label=f'Mejor época ({best})')
    plt.xlabel('Época'); plt.ylabel('Loss'); plt.title(titulo); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f'{DIR_FIG}/{nombre}_loss.png', dpi=150); plt.show()

def graficar_pred(y, yp, nombre, titulo):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for i, n in enumerate(OBJ):
        lim = max(y[:, i].max(), yp[:, i].max()) + 1
        ax[i].scatter(y[:, i], yp[:, i], s=3, alpha=.15)
        ax[i].plot([0, lim], [0, lim], 'r--', lw=1)
        ax[i].set_xlabel(f'{n} reales'); ax[i].set_ylabel(f'{n} predichos'); ax[i].set_title(n)
    fig.suptitle(titulo); plt.tight_layout(); plt.savefig(f'{DIR_FIG}/{nombre}_dispersion_prueba.png', dpi=150); plt.show()

def graficar_semana(yp_te, nombre, titulo):
    """Serie real vs predicha para la estación más usada en la primera semana completa del conjunto de prueba."""
    d = dfm[te].copy(); d[['p_ret', 'p_dev']] = yp_te
    est = attr['viajes_totales'].idxmax()
    d = d[d.estacion == est]
    lunes = d.fecha_hora[d.fecha_hora.dt.dayofweek == 0].dt.normalize().min()
    d = d[(d.fecha_hora >= lunes) & (d.fecha_hora < lunes + pd.Timedelta(days=7))]
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for a, real, pred, n in [(ax[0], 'retiros', 'p_ret', 'Retiros'), (ax[1], 'devoluciones', 'p_dev', 'Devoluciones')]:
        a.plot(d.fecha_hora, d[real], label='Real'); a.plot(d.fecha_hora, d[pred], label='Predicho')
        a.set_ylabel(n); a.legend(); a.grid(alpha=.3)
    nom = attr.loc[est, 'nombre'] if 'nombre' in attr else est
    fig.suptitle(f'{titulo} – estación {est} ({nom}), semana desde {lunes.date()}')
    plt.tight_layout(); plt.savefig(f'{DIR_FIG}/{nombre}_semana_prueba.png', dpi=150); plt.show()

def graficar_confusion(cm, nombre, titulo):
    cm = np.array(cm)
    plt.figure(figsize=(4.8, 4))
    plt.imshow(cm, cmap='Blues')
    for i in range(3):
        for j in range(3):
            plt.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.xticks(range(3), CLASES); plt.yticks(range(3), CLASES)
    plt.xlabel('Clase predicha'); plt.ylabel('Clase real'); plt.title(titulo)
    plt.tight_layout(); plt.savefig(f'{DIR_FIG}/{nombre}.png', dpi=150); plt.show()

def entrenar(modelo, nombre, titulo, Xtr, ytr, Xva, yva, lr, class_weight=None):
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss=modelo.loss_elegida)
    resumen = []
    modelo.summary(print_fn=lambda s, **k: resumen.append(s))
    print('\n'.join(resumen))
    with open(f'{DIR_RES}/{nombre}_arquitectura.txt', 'w') as f:
        f.write('\n'.join(resumen))
    cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=CONFIG['paciencia'], restore_best_weights=True)]
    t0 = time.time()
    h = modelo.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=CONFIG['epocas'], batch_size=CONFIG['batch'],
                   callbacks=cb, class_weight=class_weight, verbose=2)
    INFO_ENTRENAMIENTO[nombre] = dict(
        parametros=int(modelo.count_params()), tasa_aprendizaje=lr, batch=CONFIG['batch'],
        epocas_ejecutadas=len(h.history['loss']), mejor_epoca=int(np.argmin(h.history['val_loss'])) + 1,
        loss_entrenamiento_mejor_epoca=float(h.history['loss'][int(np.argmin(h.history['val_loss']))]),
        loss_validacion_mejor=float(min(h.history['val_loss'])), tiempo_seg=round(time.time() - t0, 1),
        historial={k: [float(x) for x in v] for k, v in h.history.items()})
    graficar_loss(h, nombre, titulo)
    return h

def predecir_conteos(modelo, Xin):
    return np.clip(modelo.predict(Xin, batch_size=4096, verbose=0), 0, None)

## 4. Línea base (métodos simples para comparar)
- **Ingenua semanal:** predice lo mismo que pasó a la misma hora de la semana anterior.
- **Promedio histórico:** promedio de entrenamiento por estación, día de la semana y hora.

In [ ]:
base_sem = dfm[['retiros_s-1', 'devol_s-1']].values
prom = dfm[tr].groupby(['estacion', 'dia_semana', 'hora'])[['retiros', 'devoluciones']].mean()
base_prom = dfm[['estacion', 'dia_semana', 'hora']].join(prom, on=['estacion', 'dia_semana', 'hora'])[['retiros', 'devoluciones']].values
METRICAS['base_ingenua_semanal'] = {'prueba': metricas_regresion(Y_reg[te], base_sem[te])}
METRICAS['base_promedio_historico'] = {'entrenamiento': metricas_regresion(Y_reg[tr], base_prom[tr]),
                                       'prueba': metricas_regresion(Y_reg[te], base_prom[te])}
mostrar_regresion('Ingenua semanal', METRICAS['base_ingenua_semanal'])
mostrar_regresion('Promedio histórico', METRICAS['base_promedio_historico'])

## 5. Entrenamiento 1 – Modelo A: MLP BP de regresión
Entrada (atributos de 2.4) → Densa 64 (ReLU) → Densa 32 (ReLU) → Salida 2 (softplus).
Pérdida MSE sobre las cantidades. En la salida se usa **softplus** en lugar de lineal para que nunca se predigan retiros negativos
(softplus(x) = log(1 + eˣ) se comporta como lineal para valores positivos y tiende a 0 para negativos).

In [ ]:
modelo_a = keras.Sequential([
    keras.Input((X.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(2, activation='softplus'),
], name='modelo_A_MLP_regresion')
modelo_a.loss_elegida = 'mse'
entrenar(modelo_a, 'modelo_A', 'Modelo A – MLP regresión', X[tr], Y_reg[tr], X[va], Y_reg[va], lr=1e-3)

pa = predecir_conteos(modelo_a, X)
METRICAS['modelo_A'] = {'entrenamiento': metricas_regresion(Y_reg[tr], pa[tr]), 'prueba': metricas_regresion(Y_reg[te], pa[te])}
mostrar_regresion('Modelo A', METRICAS['modelo_A'])
graficar_pred(Y_reg[te], pa[te], 'modelo_A', 'Modelo A – prueba: real vs predicho')
graficar_semana(pa[te], 'modelo_A', 'Modelo A')

## 6. Entrenamiento 2 – Modelo A2: MLP BP de regresión (otra topología y regularización)
Entrada → Densa 128 (ReLU) → Dropout 0,2 → Densa 64 (ReLU) → Dropout 0,2 → Densa 32 (ReLU) → Salida 2 (softplus),
con regularización L2 (1e-4) y tasa de aprendizaje 5e-4.

In [ ]:
reg = keras.regularizers.l2(1e-4)
modelo_a2 = keras.Sequential([
    keras.Input((X.shape[1],)),
    layers.Dense(128, activation='relu', kernel_regularizer=reg), layers.Dropout(0.2),
    layers.Dense(64, activation='relu', kernel_regularizer=reg), layers.Dropout(0.2),
    layers.Dense(32, activation='relu', kernel_regularizer=reg),
    layers.Dense(2, activation='softplus'),
], name='modelo_A2_MLP_regresion_profundo')
modelo_a2.loss_elegida = 'mse'
entrenar(modelo_a2, 'modelo_A2', 'Modelo A2 – MLP regresión (más profundo + regularización)',
         X[tr], Y_reg[tr], X[va], Y_reg[va], lr=5e-4)

pa2 = predecir_conteos(modelo_a2, X)
METRICAS['modelo_A2'] = {'entrenamiento': metricas_regresion(Y_reg[tr], pa2[tr]), 'prueba': metricas_regresion(Y_reg[te], pa2[te])}
mostrar_regresion('Modelo A2', METRICAS['modelo_A2'])
graficar_pred(Y_reg[te], pa2[te], 'modelo_A2', 'Modelo A2 – prueba: real vs predicho')
graficar_semana(pa2[te], 'modelo_A2', 'Modelo A2')

## 7. Entrenamiento 3 – Modelo B: MLP BP de clasificación (nivel de retiros)
Misma entrada que el modelo A → Densa 64 (ReLU) → Densa 32 (ReLU) → Salida 3 (softmax), entropía cruzada categórica.
Clases por estación: **baja** (sin retiros o ≤ percentil 33 de las horas con actividad), **media**, **alta** (> percentil 66).
Como las clases quedan desbalanceadas, se usan pesos de clase inversamente proporcionales a su frecuencia.

In [ ]:
pesos = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=Y_cls[tr])
PESOS = {i: float(w) for i, w in enumerate(pesos)}
print('Pesos de clase:', PESOS)

modelo_b = keras.Sequential([
    keras.Input((X.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax'),
], name='modelo_B_MLP_clasificacion')
modelo_b.loss_elegida = 'sparse_categorical_crossentropy'
entrenar(modelo_b, 'modelo_B', 'Modelo B – MLP clasificación', X[tr], Y_cls[tr], X[va], Y_cls[va],
         lr=1e-3, class_weight=PESOS)
INFO_ENTRENAMIENTO['modelo_B']['pesos_clase'] = PESOS

pb = modelo_b.predict(X, batch_size=4096, verbose=0).argmax(1)
METRICAS['modelo_B'] = {'entrenamiento': metricas_clasificacion(Y_cls[tr], pb[tr]),
                        'prueba': metricas_clasificacion(Y_cls[te], pb[te])}
for conj in ['entrenamiento', 'prueba']:
    m = METRICAS['modelo_B'][conj]
    print(f'\n{conj.upper()} – exactitud {100*m["exactitud"]:.2f} %')
    display(pd.DataFrame(m['por_clase']).T.round(2))
    graficar_confusion(m['matriz_confusion'], f'modelo_B_confusion_{conj}', f'Modelo B – {conj}')

## 8. Entrenamiento 4 – Modelo C: red convolucional 1D
Entrada 1: secuencia de las últimas 24 horas de la estación (retiros, devoluciones, temperatura, lluvia).
Entrada 2: calendario y clima de la hora a predecir + identificación y ubicación de la estación.
Conv1D 32 filtros (k=3, ReLU) → Conv1D 64 filtros (k=3, ReLU) → MaxPooling (2) → Aplanado → se concatena con la entrada 2 → Densa 32 (ReLU) → Salida 2 (softplus).

In [ ]:
SEQ = ['retiros', 'devoluciones', 'temperatura', 'precipitacion']
pf = p_full.sort_values(['estacion', 'fecha_hora']).reset_index(drop=True)
S = pf[SEQ].astype(float).copy()
S[['retiros', 'devoluciones']] = np.log1p(S[['retiros', 'devoluciones']])
mtr = (pf['fecha_hora'] <= fe).values
S = ((S - S[mtr].mean()) / S[mtr].std()).values.astype('float32')

ventanas = []
for est in ESTACIONES:
    arr = S[(pf['estacion'] == est).values]                       # (horas, 4)
    w = np.lib.stride_tricks.sliding_window_view(arr, 24, axis=0)  # (horas-23, 4, 24)
    ventanas.append(w[144:len(arr) - 24].transpose(0, 2, 1))       # ventana t-24..t-1 para cada t >= 168
X_seq = np.concatenate(ventanas).astype('float32')
assert len(X_seq) == len(dfm), (len(X_seq), len(dfm))

X_est, COLS_EST, _ = matriz(dfm, NUM, [])     # sin rezagos: la historia la aporta la secuencia
print('Secuencias:', X_seq.shape, '| entrada estática:', X_est.shape)

e_seq = keras.Input((24, len(SEQ)), name='ultimas_24h')
e_est = keras.Input((X_est.shape[1],), name='calendario_clima_estacion')
x = layers.Conv1D(32, 3, activation='relu')(e_seq)
x = layers.Conv1D(64, 3, activation='relu')(x)
x = layers.MaxPooling1D(2)(x)
x = layers.Flatten()(x)
z = layers.Concatenate()([x, e_est])
z = layers.Dense(32, activation='relu')(z)
salida = layers.Dense(2, activation='softplus')(z)
modelo_c = keras.Model([e_seq, e_est], salida, name='modelo_C_ConvNet1D')
modelo_c.loss_elegida = 'mse'
entrenar(modelo_c, 'modelo_C', 'Modelo C – ConvNet 1D', (X_seq[tr], X_est[tr]), Y_reg[tr],
         (X_seq[va], X_est[va]), Y_reg[va], lr=1e-3)

pc = predecir_conteos(modelo_c, (X_seq, X_est))
METRICAS['modelo_C'] = {'entrenamiento': metricas_regresion(Y_reg[tr], pc[tr]), 'prueba': metricas_regresion(Y_reg[te], pc[te])}
mostrar_regresion('Modelo C', METRICAS['modelo_C'])
graficar_pred(Y_reg[te], pc[te], 'modelo_C', 'Modelo C – prueba: real vs predicho')
graficar_semana(pc[te], 'modelo_C', 'Modelo C')

## 9. Comparación de modelos de regresión (conjunto de prueba)

In [ ]:
filas = []
for nom, clave in [('Base ingenua semanal', 'base_ingenua_semanal'), ('Base promedio histórico', 'base_promedio_historico'),
                   ('Modelo A – MLP', 'modelo_A'), ('Modelo A2 – MLP profundo', 'modelo_A2'), ('Modelo C – ConvNet 1D', 'modelo_C')]:
    for obj in OBJ:
        m = METRICAS[clave]['prueba'][obj]
        filas.append({'modelo': nom, 'objetivo': obj, 'MAE': m['error_absoluto_medio'],
                      'RMSE': m['raiz_error_cuadratico_medio'],
                      'Error relativo agregado %': m['error_relativo_agregado_pct'],
                      'Error relativo medio % (horas con demanda)': m['error_relativo_medio_pct_horas_con_demanda']})
comp = pd.DataFrame(filas).round(3)
comp.to_csv(f'{DIR_RES}/comparacion_prueba.csv', index=False)
display(comp)

pivot = comp.pivot(index='modelo', columns='objetivo', values='MAE').loc[comp.modelo.unique()]
pivot.plot.bar(figsize=(9, 4.5), rot=15)
plt.ylabel('MAE (bicicletas por hora)'); plt.title('Error absoluto medio en el conjunto de prueba'); plt.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(f'{DIR_FIG}/comparacion_mae_prueba.png', dpi=150); plt.show()

## 10. Guardado de resultados y descarga

In [ ]:
RESUMEN['entrenamientos'] = INFO_ENTRENAMIENTO
RESUMEN['metricas'] = METRICAS
RESUMEN['versiones'] = {'python_tensorflow': tf.__version__, 'pandas': pd.__version__, 'numpy': np.__version__}
with open(f'{DIR_RES}/resumen_para_informe.json', 'w', encoding='utf-8') as f:
    json.dump(RESUMEN, f, indent=2, ensure_ascii=False, default=str)

import shutil
shutil.make_archive('resultados_tp2', 'zip', DIR_RES)
shutil.make_archive('datos_tp2', 'zip', 'data')
print('Generados: resultados_tp2.zip y datos_tp2.zip')
try:
    from google.colab import files
    files.download('resultados_tp2.zip')
    files.download('datos_tp2.zip')
except Exception:
    pass